In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import string
import csv
import requests
from io import StringIO

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from wordcloud import WordCloud

from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')

pd.options.mode.chained_assignment = None
seed = 0
np.random.seed(seed)

print('Library berhasil di-import!')

Library berhasil di-import!


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\SKYL15H\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\SKYL15H\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
df = pd.read_csv('komen_XL.csv')

jumlah_komen, jumlah_kolom = df.shape
print(f'Jumlah review: {jumlah_komen}')
print(f'Jumlah kolom: {jumlah_kolom}')
print(f'Kolom: {list(df.columns)}')
print()

df.head()

Jumlah review: 576000
Jumlah kolom: 11
Kolom: ['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']



,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,52a9ccff-a3d9-41a3-8b3e-45da1d784743,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Semakin jelek kualitas jaringan XL, tiba"" sign...",1,387,9.5.0,2026-09-17 01:03:57,"Hi Kak, mohon maaf ya atas ketidaknyamanan yan...",2026-09-17 02:00:14,9.5.0
1,df316426-9c94-440e-a0c3-470e94bdd10b,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Aplikasi banyak bug, tidak stabil, sering eror...",1,69,9.5.0,2026-09-11 05:36:50,"Hi Kak, maaf ya, agar dapat dibantu cek lebih ...",2026-09-11 06:00:08,9.5.0
2,a94ab251-1eed-44d0-b6fe-ef7209c6f44d,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,⭐ 1/5 “Sering mengalami gangguan saat membeli ...,1,60,9.5.0,2026-09-09 17:45:59,"Hi Kak, mohon maaf ya atas ketidaknyamanan yan...",2026-09-09 18:00:13,9.5.0
3,a88fe295-18ae-4bad-8be1-536092dc9ad1,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Saya ubah penilaian saya karena sekarang ini a...,1,57,9.5.0,2026-09-14 08:26:40,"Hi Kak, mohon maaf ya atas kendalanya. Yuk hub...",2023-12-04 08:04:00,9.5.0
4,a8825f6d-ff7c-4817-9f54-0581455eca3f,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Sudah dua kali beli paket flexmini. Yang perta...,1,49,9.5.0,2026-09-09 17:36:08,"Hi Kak, mohon maaf ya atas ketidaknyamanan yan...",2026-09-09 18:00:17,9.5.0


In [3]:
# Deteksi nama kolom teks review
if 'content' in df.columns:
    clean_df = df[['content']].copy()
elif 'Review' in df.columns:
    clean_df = df[['Review']].copy()
    clean_df.rename(columns={'Review': 'content'}, inplace=True)
else:
    # Gunakan kolom pertama sebagai fallback
    clean_df = df.iloc[:, [0]].copy()
    clean_df.columns = ['content']

# Hapus baris yang null/kosong
clean_df = clean_df.dropna(subset=['content'])
clean_df = clean_df[clean_df['content'].str.strip() != '']

# Hapus duplikat
jumlah_sebelum = len(clean_df)
clean_df = clean_df.drop_duplicates()
jumlah_setelah = len(clean_df)

print(f'Jumlah sebelum hapus duplikat: {jumlah_sebelum}')
print(f'Jumlah setelah hapus duplikat: {jumlah_setelah}')
print(f'Data duplikat dihapus: {jumlah_sebelum - jumlah_setelah}')

clean_df.reset_index(drop=True, inplace=True)
clean_df.head()

Jumlah sebelum hapus duplikat: 575999
Jumlah setelah hapus duplikat: 380290
Data duplikat dihapus: 195709


,content
0,"Semakin jelek kualitas jaringan XL, tiba"" sign..."
1,"Aplikasi banyak bug, tidak stabil, sering eror..."
2,⭐ 1/5 “Sering mengalami gangguan saat membeli ...
3,Saya ubah penilaian saya karena sekarang ini a...
4,Sudah dua kali beli paket flexmini. Yang perta...


In [4]:
def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text)
    text = re.sub(r'#[A-Za-z0-9]+', '', text)
    text = re.sub(r'RT[\s]', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[0-9]+', '', text)
    text = re.sub(r'[^\w\s]', '', text)

    text = text.replace('\n', ' ')
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.strip(' ')
    return text

def casefoldingText(text):
    text = text.lower()
    return text

def tokenizeText(text):
    text = word_tokenize(text)
    return text

def filteringText(text):
    listStopwords = set(stopwords.words('indonesian'))
    listStopwords.update(stopwords.words('english'))
    listStopwords.update(['iya','yaa','gak','nya','na','sih','ku',"di","ga","ya","gaa","loh","kah","woi","woii","woy"])
    filtered = []

    for txt in text:
        if txt not in listStopwords:
            filtered.append(txt)
    text = filtered
    return text

def stemmingText(text):
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()
    words = text.split()
    stemmed_words = [stemmer.stem(word) for word in words]
    stemmed_text = ' '.join(stemmed_words)
    return stemmed_text

def toSentence(list_words):
    sentence = ' '.join(word for word in list_words)
    return sentence

print('Fungsi preprocessing berhasil didefinisikan!')

Fungsi preprocessing berhasil didefinisikan!


In [5]:
# Kamus slangwords untuk normalisasi kata
slangwords = {
    "@": "di", "abis": "habis", "wtb": "beli", "masi": "masih",
    "wts": "jual", "wtt": "tukar", "bgt": "banget", "maks": "maksimal"
}

def fix_slangwords(text):
    words = text.split()
    fixed_words = []

    for word in words:
        if word.lower() in slangwords:
            fixed_words.append(slangwords[word.lower()])
        else:
            fixed_words.append(word)

    fixed_text = ' '.join(fixed_words)
    return fixed_text

print('Kamus slangwords berhasil dimuat!')

Kamus slangwords berhasil dimuat!


In [6]:
print('Menjalankan preprocessing...')

clean_df['text_clean'] = clean_df['content'].apply(cleaningText)
print('  ✓ Cleaning selesai')

clean_df['text_casefoldingText'] = clean_df['text_clean'].apply(casefoldingText)
print('  ✓ Casefolding selesai')

clean_df['text_slangwords'] = clean_df['text_casefoldingText'].apply(fix_slangwords)
print('  ✓ Fix slangwords selesai')

clean_df['text_tokenizingText'] = clean_df['text_slangwords'].apply(tokenizeText)
print('  ✓ Tokenizing selesai')

clean_df['text_stopword'] = clean_df['text_tokenizingText'].apply(filteringText)
print('  ✓ Stopword removal selesai')

clean_df['text_akhir'] = clean_df['text_stopword'].apply(toSentence)
print('  ✓ Konversi ke kalimat selesai')

print(f'\nPreprocessing selesai! ({len(clean_df)} baris)')
clean_df[['content', 'text_akhir']].head(10)

Menjalankan preprocessing...
  ✓ Cleaning selesai
  ✓ Casefolding selesai
  ✓ Fix slangwords selesai
  ✓ Tokenizing selesai
  ✓ Stopword removal selesai
  ✓ Konversi ke kalimat selesai

Preprocessing selesai! (380290 baris)


,content,text_akhir
0,"Semakin jelek kualitas jaringan XL, tiba"" sign...",jelek kualitas jaringan xl signal hilang paket...
1,"Aplikasi banyak bug, tidak stabil, sering eror...",aplikasi bug stabil eror tampilan reward suka ...
2,⭐ 1/5 “Sering mengalami gangguan saat membeli ...,mengalami gangguan membeli paket pulsa penukar...
3,Saya ubah penilaian saya karena sekarang ini a...,ubah penilaian app bermasalah buka alasan jari...
4,Sudah dua kali beli paket flexmini. Yang perta...,kali beli paket flexmini flexmini mingguan taw...
5,sudah 2 minggu wifi bermasalah jaringan tanpa ...,minggu wifi bermasalah jaringan koneksi aduan ...
6,"susah sekali untuk membeli paket, pulsa cukup ...",susah membeli paket pulsa beli paket gangguan ...
7,Tidak ada pemberitahuan dan kejelasan sebelumn...,pemberitahuan kejelasan pemeliharaan jaringan ...
8,"halo selamat malam, saya barusan beli kuota fl...",halo selamat malam barusan beli kuota flexmax ...
9,Saya kurang suka sama sistemnya padahal tadiny...,suka sistemnya udah level platinum berubah lev...


In [7]:
# Download lexicon positif
lexicon_positive = dict()

response = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_positive.csv')

if response.status_code == 200:
    reader = csv.reader(StringIO(response.text), delimiter=',')
    for row in reader:
        lexicon_positive[row[0]] = int(row[1])
    print(f'Lexicon positif berhasil dimuat: {len(lexicon_positive)} kata')
else:
    print('Gagal mengambil data lexicon positive')

# Download lexicon negatif
lexicon_negative = dict()

response = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_negative.csv')

if response.status_code == 200:
    reader = csv.reader(StringIO(response.text), delimiter=',')
    for row in reader:
        lexicon_negative[row[0]] = int(row[1])
    print(f'Lexicon negatif berhasil dimuat: {len(lexicon_negative)} kata')
else:
    print('Gagal mengambil data lexicon negative')

Lexicon positif berhasil dimuat: 3609 kata
Lexicon negatif berhasil dimuat: 6607 kata


In [8]:
def sentiment_analysis_lexicon_indonesia(text):
    score = 0

    for word in text:
        if (word in lexicon_positive):
            score = score + lexicon_positive[word]

    for word in text:
        if (word in lexicon_negative):
            score = score + lexicon_negative[word]

    polarity=''

    if (score >= 0):
        polarity = 'positive'
    elif (score < 0):
        polarity = 'negative'
    return score, polarity

In [9]:
# Terapkan analisis sentimen
results = clean_df['text_stopword'].apply(sentiment_analysis_lexicon_indonesia)
results = list(zip(*results))
clean_df['polarity_score'] = results[0]
clean_df['polarity'] = results[1]

print('Distribusi Sentimen:')
print(clean_df['polarity'].value_counts())
print()
clean_df[['content', 'text_akhir', 'polarity_score', 'polarity']].head(10)

Distribusi Sentimen:
polarity
negative    200558
positive    179732
Name: count, dtype: int64



,content,text_akhir,polarity_score,polarity
0,"Semakin jelek kualitas jaringan XL, tiba"" sign...",jelek kualitas jaringan xl signal hilang paket...,-16,negative
1,"Aplikasi banyak bug, tidak stabil, sering eror...",aplikasi bug stabil eror tampilan reward suka ...,-12,negative
2,⭐ 1/5 “Sering mengalami gangguan saat membeli ...,mengalami gangguan membeli paket pulsa penukar...,6,positive
3,Saya ubah penilaian saya karena sekarang ini a...,ubah penilaian app bermasalah buka alasan jari...,-22,negative
4,Sudah dua kali beli paket flexmini. Yang perta...,kali beli paket flexmini flexmini mingguan taw...,-6,negative
5,sudah 2 minggu wifi bermasalah jaringan tanpa ...,minggu wifi bermasalah jaringan koneksi aduan ...,8,positive
6,"susah sekali untuk membeli paket, pulsa cukup ...",susah membeli paket pulsa beli paket gangguan ...,-16,negative
7,Tidak ada pemberitahuan dan kejelasan sebelumn...,pemberitahuan kejelasan pemeliharaan jaringan ...,-24,negative
8,"halo selamat malam, saya barusan beli kuota fl...",halo selamat malam barusan beli kuota flexmax ...,-5,negative
9,Saya kurang suka sama sistemnya padahal tadiny...,suka sistemnya udah level platinum berubah lev...,3,positive


In [10]:
# Vektorisasi teks menggunakan TF-IDF
x = clean_df['text_akhir']
y = clean_df['polarity']

tfidf = TfidfVectorizer(max_features=200, min_df=17, max_df=0.8)
X_tfidf = tfidf.fit_transform(x)

features_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out())
print(f'Shape fitur TF-IDF: {features_df.shape}')
features_df.head()

Shape fitur TF-IDF: (380290, 200)


,aja,aktif,ambil,aneh,apk,aplikasi,aplikasinya,app,bagus,banget,...,ulang,unlimited,update,utama,voucher,wifi,xl,xtra,yg,youtube
0,0.000000,0.0,0.00000,0.0,0.0,0.000000,0.0,0.000000,0.135979,0.000000,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.178994,0.0,0.0,0.0
1,0.000000,0.0,0.00000,0.0,0.0,0.385053,0.0,0.000000,0.151090,0.000000,...,0.000000,0.0,0.342855,0.0,0.0,0.0,0.099442,0.0,0.0,0.0
2,0.000000,0.0,0.00000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
3,0.000000,0.0,0.00000,0.0,0.0,0.000000,0.0,0.300428,0.190647,0.000000,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
4,0.125174,0.0,0.19923,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.122109,...,0.198574,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0


In [11]:
# Split data training dan testing
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

print(f'Data training: {X_train.shape[0]} sampel')
print(f'Data testing: {X_test.shape[0]} sampel')

Data training: 304232 sampel
Data testing: 76058 sampel


In [12]:
# Training Random Forest
random_forest = RandomForestClassifier(random_state=42)
random_forest.fit(X_train.toarray(), y_train)

# Prediksi
y_pred_train_rf = random_forest.predict(X_train.toarray())
y_pred_test_rf = random_forest.predict(X_test.toarray())

# Evaluasi
accuracy_train_rf = accuracy_score(y_pred_train_rf, y_train)
accuracy_test_rf = accuracy_score(y_pred_test_rf, y_test)

print(f'Random Forest - accuracy_train: {accuracy_train_rf:.4f}')
print(f'Random Forest - accuracy_test: {accuracy_test_rf:.4f}')

Random Forest - accuracy_train: 0.9587
Random Forest - accuracy_test: 0.8771
